# Silver Layer ELT — Data Transformation & Cleaning Procedure

## Overview
This Databricks PySpark notebook implements the Extract, Load, and Transform (ELT) pipeline using scalable, production-grade PySpark DataFrame operations on Delta Lake.

### Ingestion & Transformation Highlights:
1. **`silver.crm_cust_info`**:
   - Eliminates null customer IDs (`cst_id IS NOT NULL`).
   - Deduplicates records on `cst_id` retaining the most recent record via `ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) = 1`.
   - Strips leading and trailing whitespace from customer names (`TRIM`).
   - Standardizes marital status (`'M'` -> `'Married'`, `'S'` -> `'Single'`, otherwise `'unknown'`).
   - Standardizes gender codes (`'F'` -> `'Female'`, `'M'` -> `'Male'`, otherwise `'unknown'`).
   - Adds audit timestamp `dwh_create_date = current_timestamp()`.

2. **`silver.crm_prd_info`**:
   - Dissects composite product key: extracts `cat_id = REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_')` and clean `prd_key = SUBSTRING(prd_key, 7)`.
   - Imputes null product cost to `0` (`COALESCE(prd_cost, 0)`).
   - Maps abbreviated product lines: `'M'` -> `'Mountain'`, `'R'` -> `'Road'`, `'S'` -> `'Other Sales'`, `'T'` -> `'Touring'`, otherwise `'unknown'`.
   - Computes Slowly Changing Dimension (SCD) Type 2 expiry dates (`prd_end_dt`) using `LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt) - INTERVAL 1 DAY`.
   - Adds audit timestamp `dwh_create_date = current_timestamp()`.

3. **`silver.crm_sales_details`**:
   - Validates and parses integer `YYYYMMDD` dates (`sls_order_dt`, `sls_ship_dt`, `sls_due_dt`) to native Spark `DateType` (setting invalid lengths or zero dates to `NULL`).
   - Enforces sales amount integrity: recalculates `sls_sales = sls_quantity * ABS(sls_price)` if missing, zero, negative, or inconsistent.
   - Derives unit price: `sls_price = sls_sales / NULLIF(sls_quantity, 0)` if missing or non-positive.
   - Adds audit timestamp `dwh_create_date = current_timestamp()`.

4. **`silver.erp_cust_az12`**:
   - Strips `'NAS'` prefix from `cid` to align with CRM keys (`SUBSTRING(cid, 4)`).
   - Invalidates future birthdates (`bdate > current_date()` set to `NULL`).
   - Standardizes casing and whitespace for gender (`'Male'`, `'Female'`, or `'unknown'`).
   - Adds audit timestamp `dwh_create_date = current_timestamp()`.

5. **`silver.erp_loc_a101`**:
   - Normalizes customer identifiers by stripping hyphens (`REPLACE(cid, '-', '')`).
   - Standardizes country codes and names (`'DE'` -> `'Germany'`, `'US'`/`'USA'` -> `'United States'`, blanks -> `'unknown'`).
   - Adds audit timestamp `dwh_create_date = current_timestamp()`.

6. **`silver.erp_px_cat_g1v2`**:
   - Standard product hierarchy passthrough with audit timestamp `dwh_create_date = current_timestamp()`.

All tables are atomically written to Delta Lake using `.mode("overwrite").saveAsTable(...)` ensuring idempotent batch execution.

In [ ]:
# Databricks notebook source
# COMMAND ----------
import time
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, trim, upper, when, lit, coalesce, substring, length,
    regexp_replace, to_date, abs as spark_abs, current_date,
    current_timestamp, lead, date_sub, expr, row_number
)

# Display helper fallback for non-Databricks interactive sessions
try:
    display
except NameError:
    def display(df, n=20):
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

# Start batch execution timer
batch_start_time = time.time()
print("================================================")
print("Loading Silver Layer")
print("================================================")

## 1. Transform CRM Customer Info (`silver.crm_cust_info`)
Read raw staging data from `bronze.crm_cust_info`, filter null IDs, deduplicate by latest create date, trim names, and standardize marital status and gender.

In [ ]:
# COMMAND ----------
start_time = time.time()
print("------------------------------------------------")
print(">> Transforming & Loading Table: silver.crm_cust_info")

# Deduplication window by customer ID ordered by creation date descending
cust_window = Window.partitionBy("cst_id").orderBy(col("cst_create_date").desc())

df_bronze_cust = spark.table("bronze.crm_cust_info").filter(col("cst_id").isNotNull())

df_silver_cust = (
    df_bronze_cust
    .withColumn("flag_last", row_number().over(cust_window))
    .filter(col("flag_last") == 1)
    .select(
        col("cst_id"),
        col("cst_key"),
        trim(col("cst_firstname")).alias("cst_firstname"),
        trim(col("cst_lastname")).alias("cst_lastname"),
        when(upper(trim(col("cst_marital_status"))) == "M", "Married")
            .when(upper(trim(col("cst_marital_status"))) == "S", "Single")
            .otherwise("unknown").alias("cst_marital_status"),
        when(upper(trim(col("cst_gndr"))) == "F", "Female")
            .when(upper(trim(col("cst_gndr"))) == "M", "Male")
            .otherwise("unknown").alias("cst_gndr"),
        col("cst_create_date").cast("date").alias("cst_create_date"),
        current_timestamp().alias("dwh_create_date")
    )
)

# Atomic overwrite to Delta table
(
    df_silver_cust.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.crm_cust_info")
)

cust_duration = time.time() - start_time
print(f">> Load Duration: {cust_duration:.2f} seconds | Target: silver.crm_cust_info")

## 2. Transform CRM Product Info (`silver.crm_prd_info`)
Read raw staging data from `bronze.crm_prd_info`, extract category ID and clean product key, impute null cost to 0, map product line codes, and compute SCD Type 2 expiry dates (`prd_end_dt`).

In [ ]:
# COMMAND ----------
start_time = time.time()
print("------------------------------------------------")
print(">> Transforming & Loading Table: silver.crm_prd_info")

# Extract clean product key and category ID
df_bronze_prd = spark.table("bronze.crm_prd_info")

df_prd_staged = (
    df_bronze_prd
    .withColumn("cat_id", regexp_replace(substring(col("prd_key"), 1, 5), "-", "_"))
    .withColumn("clean_prd_key", substring(col("prd_key"), 7, length(col("prd_key"))))
    .withColumn("prd_start_dt_date", col("prd_start_dt").cast("date"))
)

# SCD Type 2 Window partitioned by clean_prd_key ordered by start date
prd_window = Window.partitionBy("clean_prd_key").orderBy("prd_start_dt_date")

df_silver_prd = (
    df_prd_staged
    .withColumn(
        "prd_end_dt",
        date_sub(lead(col("prd_start_dt_date")).over(prd_window), 1)
    )
    .select(
        col("prd_id"),
        col("cat_id"),
        col("clean_prd_key").alias("prd_key"),
        trim(col("prd_nm")).alias("prd_nm"),
        coalesce(col("prd_cost"), lit(0)).alias("prd_cost"),
        when(upper(trim(col("prd_line"))) == "M", "Mountain")
            .when(upper(trim(col("prd_line"))) == "R", "Road")
            .when(upper(trim(col("prd_line"))) == "S", "Other Sales")
            .when(upper(trim(col("prd_line"))) == "T", "Touring")
            .otherwise("unknown").alias("prd_line"),
        col("prd_start_dt_date").alias("prd_start_dt"),
        col("prd_end_dt"),
        current_timestamp().alias("dwh_create_date")
    )
)

# Atomic overwrite to Delta table
(
    df_silver_prd.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.crm_prd_info")
)

prd_duration = time.time() - start_time
print(f">> Load Duration: {prd_duration:.2f} seconds | Target: silver.crm_prd_info")

## 3. Transform CRM Sales Details (`silver.crm_sales_details`)
Read raw staging data from `bronze.crm_sales_details`, parse integer dates (`YYYYMMDD`) to native `DateType`, enforce sales amount reconciliation (`sls_sales = sls_quantity * ABS(sls_price)`), and derive missing prices.

In [ ]:
# COMMAND ----------
start_time = time.time()
print("------------------------------------------------")
print(">> Transforming & Loading Table: silver.crm_sales_details")

df_bronze_sales = spark.table("bronze.crm_sales_details")

df_silver_sales = (
    df_bronze_sales
    .withColumn("order_dt_str", col("sls_order_dt").cast("string"))
    .withColumn("ship_dt_str", col("sls_ship_dt").cast("string"))
    .withColumn("due_dt_str", col("sls_due_dt").cast("string"))
    .select(
        col("sls_ord_num"),
        col("sls_prd_key"),
        col("sls_cust_id"),
        # Date parsing: null if 0 or string length is not 8
        when((col("sls_order_dt") == 0) | (length(col("order_dt_str")) != 8), lit(None))
            .otherwise(to_date(col("order_dt_str"), "yyyyMMdd")).alias("sls_order_dt"),
        when((col("sls_ship_dt") == 0) | (length(col("ship_dt_str")) != 8), lit(None))
            .otherwise(to_date(col("ship_dt_str"), "yyyyMMdd")).alias("sls_ship_dt"),
        when((col("sls_due_dt") == 0) | (length(col("due_dt_str")) != 8), lit(None))
            .otherwise(to_date(col("due_dt_str"), "yyyyMMdd")).alias("sls_due_dt"),
        # Sales amount validation: recalculate if null, non-positive, or inconsistent with quantity * abs(price)
        when(
            col("sls_sales").isNull() | (col("sls_sales") <= 0) | (col("sls_sales") != col("sls_quantity") * spark_abs(col("sls_price"))),
            col("sls_quantity") * spark_abs(col("sls_price"))
        ).otherwise(col("sls_sales")).alias("sls_sales"),
        col("sls_quantity"),
        # Unit price derivation: derive from sales / quantity if price is null or non-positive
        when(
            col("sls_price").isNull() | (col("sls_price") <= 0),
            (col("sls_sales") / when(col("sls_quantity") == 0, lit(None)).otherwise(col("sls_quantity"))).cast("int")
        ).otherwise(col("sls_price")).alias("sls_price"),
        current_timestamp().alias("dwh_create_date")
    )
)

# Atomic overwrite to Delta table
(
    df_silver_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.crm_sales_details")
)

sales_duration = time.time() - start_time
print(f">> Load Duration: {sales_duration:.2f} seconds | Target: silver.crm_sales_details")

## 4. Transform ERP Customer Demographics (`silver.erp_cust_az12`)
Read raw staging data from `bronze.erp_cust_az12`, strip `'NAS'` prefix from `cid`, set future birthdates to `NULL`, and standardize gender strings.

In [ ]:
# COMMAND ----------
start_time = time.time()
print("------------------------------------------------")
print(">> Transforming & Loading Table: silver.erp_cust_az12")

df_bronze_cust_az12 = spark.table("bronze.erp_cust_az12")

df_silver_cust_az12 = (
    df_bronze_cust_az12
    .select(
        # Strip 'NAS' prefix if present
        when(col("cid").startswith("NAS"), substring(col("cid"), 4, length(col("cid"))))
            .otherwise(col("cid")).alias("cid"),
        # Invalidate future birthdates to NULL
        when(col("bdate") > current_date(), lit(None))
            .otherwise(col("bdate")).alias("bdate"),
        # Standardize gender
        when(upper(trim(col("gen"))).isin("M", "MALE"), "Male")
            .when(upper(trim(col("gen"))).isin("F", "FEMALE"), "Female")
            .otherwise("unknown").alias("gen"),
        current_timestamp().alias("dwh_create_date")
    )
)

# Atomic overwrite to Delta table
(
    df_silver_cust_az12.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.erp_cust_az12")
)

cust_az12_duration = time.time() - start_time
print(f">> Load Duration: {cust_az12_duration:.2f} seconds | Target: silver.erp_cust_az12")

## 5. Transform ERP Customer Locations (`silver.erp_loc_a101`)
Read raw staging data from `bronze.erp_loc_a101`, strip hyphens from customer keys (`cid`), and standardize country names.

In [ ]:
# COMMAND ----------
start_time = time.time()
print("------------------------------------------------")
print(">> Transforming & Loading Table: silver.erp_loc_a101")

df_bronze_loc = spark.table("bronze.erp_loc_a101")

df_silver_loc = (
    df_bronze_loc
    .select(
        # Remove hyphens from customer ID
        regexp_replace(col("cid"), "-", "").alias("cid"),
        # Country name standardization
        when(trim(col("cntry")) == "DE", "Germany")
            .when(trim(col("cntry")).isin("US", "USA"), "United States")
            .when((col("cntry").isNull()) | (trim(col("cntry")) == ""), "unknown")
            .otherwise(trim(col("cntry"))).alias("cntry"),
        current_timestamp().alias("dwh_create_date")
    )
)

# Atomic overwrite to Delta table
(
    df_silver_loc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.erp_loc_a101")
)

loc_duration = time.time() - start_time
print(f">> Load Duration: {loc_duration:.2f} seconds | Target: silver.erp_loc_a101")

## 6. Transform ERP Product Categories (`silver.erp_px_cat_g1v2`)
Read raw staging data from `bronze.erp_px_cat_g1v2` and load into `silver.erp_px_cat_g1v2` with audit timestamp.

In [ ]:
# COMMAND ----------
start_time = time.time()
print("------------------------------------------------")
print(">> Transforming & Loading Table: silver.erp_px_cat_g1v2")

df_bronze_cat = spark.table("bronze.erp_px_cat_g1v2")

df_silver_cat = (
    df_bronze_cat
    .select(
        col("id"),
        trim(col("cat")).alias("cat"),
        trim(col("subcat")).alias("subcat"),
        trim(col("maintenance")).alias("maintenance"),
        current_timestamp().alias("dwh_create_date")
    )
)

# Atomic overwrite to Delta table
(
    df_silver_cat.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.erp_px_cat_g1v2")
)

cat_duration = time.time() - start_time
print(f">> Load Duration: {cat_duration:.2f} seconds | Target: silver.erp_px_cat_g1v2")

## 7. Execution Summary & Layer Validation
Output total pipeline execution metrics and verify row counts and sample records across all 6 Silver tables.

In [ ]:
# COMMAND ----------
batch_total_duration = time.time() - batch_start_time
print("==========================================")
print("Loading Silver Layer is Completed")
print(f"   - Total Load Duration: {batch_total_duration:.2f} seconds")
print("==========================================")

# Verify row counts and display sample data from each silver table
silver_tables = [
    "silver.crm_cust_info",
    "silver.crm_prd_info",
    "silver.crm_sales_details",
    "silver.erp_loc_a101",
    "silver.erp_cust_az12",
    "silver.erp_px_cat_g1v2"
]

for tbl in silver_tables:
    df_tbl = spark.table(tbl)
    cnt = df_tbl.count()
    print(f"Table: {tbl:<26} | Record Count: {cnt:>6}")
    display(df_tbl.limit(5))